# 💰 第七课 · 挖矿经济学 & 攻击

## 本课目标

- 理解矿工的盈亏模型
- 理解 51% 攻击是什么、为什么在 BTC 上没发生过
- 通过蒙特卡洛模拟感受 51% 攻击的成功率
- 了解 MEV 这个新概念

> 这一课把密码学、博弈论和经济学串在一起。

## 1. 矿工是生意人

矿工每天问自己一个简单问题：**我赚的钱够付电费吗？**

```
日利润 = 日收入 - 日电费

日收入 = 区块奖励 × 144 × 我的算力占比 × BTC汇率
日电费 = (矿机功耗kW) × 24 × 电价
```

**关键变量**（任何一个变了，盈亏可能翻转）：
- BTC 价格（最直接影响）
- 全网算力（决定你的份额）
- 电价（矿场选址的决定因素）
- 矿机能效（W/TH，越低越好）

这就是为什么矿场都在电最便宜的地方——
四川的水电、新疆的煤电、哈萨克斯坦的弃电。

In [ ]:
def mining_profit(
    btc_price: float,       # BTC 价格（美元）
    network_hashrate: float, # 全网算力 (EH/s)
    your_hashrate: float,    # 你的算力 (TH/s)
    power_w: float,          # 功耗 (瓦)
    elec_cost: float,        # 电价 ($/kWh)
    block_reward: float = 3.125,
):
    # 每天全网总收入
    daily_btc_total = 144 * block_reward
    daily_usd_total = daily_btc_total * btc_price

    # 你的份额
    your_share = your_hashrate / (network_hashrate * 1e6)  # EH→TH
    your_btc = daily_btc_total * your_share
    your_usd = your_btc * btc_price

    # 你的电费
    daily_kwh = (power_w / 1000) * 24
    daily_cost = daily_kwh * elec_cost

    profit = your_usd - daily_cost

    print(f"BTC价格: ${btc_price:,.0f}")
    print(f"电价:    ${elec_cost:.3f}/kWh")
    print(f"你的份额: {your_share*100:.8f}%")
    print(f"日收入:   ${your_usd:.2f}")
    print(f"日电费:   ${daily_cost:.2f}")
    print(f"日利润:   ${profit:.2f}")
    return profit

print("=" * 50)
print("场景 1: 🟢 牛市 + 便宜电")
mining_profit(
    btc_price=100_000,
    network_hashrate=600,  # 600 EH/s
    your_hashrate=200,     # 200 TH/s (一台 S21)
    power_w=3500,
    elec_cost=0.05,
)

print()
print("=" * 50)
print("场景 2: 🔴 熊市 + 贵电")
mining_profit(
    btc_price=20_000,
    network_hashrate=600,
    your_hashrate=200,
    power_w=3500,
    elec_cost=0.15,
)

print()
print("=" * 50)
print("场景 3: 💀 你的 MacBook (幻想)")
mining_profit(
    btc_price=1_000_000,
    network_hashrate=600,
    your_hashrate=0.00001,  # 10 KH/s (高估了)
    power_w=30,
    elec_cost=0.10,
)  

## 2. 51% 攻击

**定义**：如果你控制了超过 50% 的全网算力，你可以：

✅ **可以做到的**：
- **双花**：花钱 → 等确认 → 秘密挖更长链 → 广播 → 推翻之前的交易
- **审查交易**：不打包某些人的交易

❌ **做不到的**：
- 偷别人的币（没有私钥）
- 凭空造币（违反共识规则，其他节点拒绝）
- 更改历史（即使 51% 攻击也只改你自己的区块）

**为什么没人攻击比特币？**
- 硬件成本：300+ EH/s × $30/TH ≈ **$90 亿**
- 攻击导致 BTC 崩盘，你的 $90 亿矿机变废铁
- 博弈论结论：诚实挖矿的长期收益 >> 攻击的一次性收益

这不是"道德高尚"，是"数学算下来划不来"。

## 3. 蒙特卡洛模拟：51% 成功率

下面用蒙特卡洛方法模拟：给定攻击者算力占比，
双花攻击成功率是多少。

In [ ]:
import random

def simulate_double_spend(attacker_pct: float, confirmations: int = 6, trials: int = 500):
    # 蒙特卡洛模拟双花攻击成功率
    # attacker_pct: 攻击者算力占比 (0~1)
    # confirmations: 需要多少确认（比特币默认为 6）
    # trials: 模拟次数

    successes = 0

    for _ in range(trials):
        honest = 0
        attacker = 0

        # 阶段1: 诚实链挖到 N 个确认
        while honest < confirmations:
            if random.random() < attacker_pct:
                attacker += 1
            else:
                honest += 1

        # 阶段2: 攻击者追赶（需要超过诚实链）
        deficit = honest - attacker
        while deficit > 0 and honest < 200:  # honest<200 是上限保护
            if random.random() < attacker_pct:
                attacker += 1
            else:
                honest += 1
            deficit = honest - attacker

        if deficit <= 0:
            successes += 1

    return successes / trials

# 测试不同算力占比
print("攻击成功率 vs 算力占比 (6 次确认):")
print("-" * 45)
for pct in [0.10, 0.20, 0.30, 0.40, 0.45, 0.49, 0.51, 0.55]:
    rate = simulate_double_spend(pct, confirmations=6, trials=500)
    bar = "█" * int(rate * 40)
    print(f"  {pct*100:5.1f}% 算力: {rate*100:5.1f}% {bar}")

print()
print("💡 即使有 51% 算力，成功率也 ≈ 70%，不是 100%")
print("💡 低于 45% 几乎不可能成功")  

## 4. 自私挖矿

2013 年，Ittay Eyal 提出了"自私挖矿"策略：

1. 挖到块后**不广播**，秘密继续挖下一个
2. 当诚实链快追上时，一口气放出多个块
3. 理论上 < 50% 算力也可能获得超额收益

**为什么没有实际发生？**
- 需要精准的网络时序控制（控制什么时候放块）
- 以太坊的 Uncle Block 机制部分缓解了
- 现实中矿工更愿意稳定拿钱，而不是冒险

学术界认为这是一个理论漏洞，但实践中不构成威胁。

## 5. MEV — 最大的可提取价值

MEV 主要是以太坊生态的概念，但原理适用于所有区块链：

矿工/验证者通过**重排、插入、审查**交易来获取额外利润：
- **抢跑**：看到别人要买入 → 自己先买 → 高价卖回
- **三明治攻击**：在大买单前后各放一笔交易
- **清算套利**：抢先清算 DeFi 坏账拿奖励

MEV 暴露了一个哲学问题：
**即使共识机制是安全的，经济激励仍可能产生不良行为。**

现在有 Flashbots 等协议试图将 MEV 公开化竞争化，
让矿工通过拍卖而非抢先交易来获取 MEV。

这是区块链领域的前沿研究方向。

## 本课小结

| 要点 | 记住 |
|------|------|
| 挖矿盈亏 | 取决于币价、电价、算力占比 |
| 51% 攻击 | 成本 >$90亿，经济上不可行 |
| 双花 | 6 确认后成功率极低，比特币最安全的确认数 |
| 自私挖矿 | 理论存在，实际未发生 |
| MEV | 激励设计的新前沿 |

**安全 = 密码学 + 博弈论 + 经济激励**

**下一课**：最后一课——真实数据、硬件进化、比特币的未来。